# Stage1 09:45 跨 ETF 响应：双接收任务必要信息门

冻结原39输入与一项新坐标，在同一来源已知日期比较匹配 Ridge 控制和候选；复用历史仅作开发反馈。

In [ ]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import _ssh,archive_nas_recorder,_settings
root=pathlib.Path.cwd();card=root/'research/stage1_0945_cross_response_dual_h5_information_20261023'
plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal']
assert sha(root/src['coordinate_card']/'summary.json')==src['coordinate_summary_sha256']
assert sha(root/src['coordinate_card']/'coordinate_result.json')==src['coordinate_result_sha256']
assert sha(root/src['intraday_provider']/'qlib_store_manifest.json')==src['intraday_manifest_sha256']
assert sha(root/src['original39_dataset_card']/'summary.json')==src['original39_summary_sha256']
assert sha(root/'src/analysis/seed_disagreement_information.py')==src['bootstrap_module_sha256']
original_card=root/src['original39_dataset_card'];dataset_path=original_card/'qlib_recorder'/src['original39_experiment_id']/src['original39_recorder_id']/'artifacts/inputs/dataset'
assert sha(dataset_path)==src['original39_dataset_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/src['intraday_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
fields=['$open','$close','$is_complete','$observed_1m_count','$expected_1m_count']
raw=D.features(['SOXL','SOXX'],fields,start_time='2018-10-12',end_time='2026-10-07 16:00:00',freq='15min').reset_index()
raw['datetime']=pd.to_datetime(raw.datetime);first=raw.loc[raw.datetime.dt.strftime('%H:%M').eq('09:45')].copy();first['signal_date']=first.datetime.dt.normalize()
first['valid']=np.isfinite(first['$open'])&np.isfinite(first['$close'])&(first['$open']>0)&(first['$close']>0)&first['$is_complete'].eq(1)&first['$observed_1m_count'].eq(15)&first['$expected_1m_count'].eq(15)
first['response']=(first['$close']/first['$open']-1).where(first.valid)
pair=first.pivot(index='signal_date',columns='instrument',values='response').reindex(columns=['SOXL','SOXX'])
coordinate=pair.SOXL-3*pair.SOXX
assert coordinate.loc['2018-10-12':'2022-12-30'].notna().sum()==1061 and coordinate.loc['2023-01-03':'2026-10-07'].notna().sum()==944
with R.uri_context(str(original_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['original39_experiment_id'],recorder_id=src['original39_recorder_id'])
 assert rec.status=='FINISHED';dataset=rec.load_object('inputs/dataset')
train_all=dataset.prepare('train',col_set=['feature','label','meta']);held_all=dataset.prepare('held',col_set=['feature','label','meta'])
feature_names=list(train_all['feature'].columns);assert len(feature_names)==39
train_dates=pd.DatetimeIndex(train_all.index.get_level_values('datetime')).normalize();held_dates=pd.DatetimeIndex(held_all.index.get_level_values('datetime')).normalize()
train_source=coordinate.reindex(train_dates).to_numpy(float);held_source=coordinate.reindex(held_dates).to_numpy(float)
train_mask=(train_dates>=pd.Timestamp('2018-10-15'))&(train_dates<=pd.Timestamp('2022-12-30'))&train_all[('meta','train_mature')].to_numpy(bool)&np.isfinite(train_source)
held_mask=(held_dates>=pd.Timestamp('2023-01-03'))&(held_dates<=pd.Timestamp('2026-10-07'))&held_all[('meta','held_mature')].to_numpy(bool)&np.isfinite(held_source)
tr=train_all.iloc[np.flatnonzero(train_mask)];he=held_all.iloc[np.flatnonzero(held_mask)]
ct=train_source[train_mask];ch=held_source[held_mask]
assert len(tr)>=1000 and len(he)==940 and len(ct)==len(tr) and len(ch)==len(he)
assert not np.isfinite(coordinate.loc['2020-03-16']) and pd.Timestamp('2020-03-16') not in pd.DatetimeIndex(tr.index.get_level_values('datetime')).normalize()
assert len(held_all)==944
xtr=tr['feature'][feature_names].to_numpy(float);xhe=he['feature'][feature_names].to_numpy(float)
xtr_new=np.column_stack([xtr,ct]);xhe_new=np.column_stack([xhe,ch])
preflight={'status':'matched_original39_plus_one_source_known_H5_preflight_passed','plan_sha256':sha(card/'plan.json'),'train_mature_source_known':len(tr),'held_mature_source_known':len(he),'full_held_calendar':len(held_all),'excluded_source_unknown_training_date':'2020-03-16','source_Recorder_object_loads':1,'Qlib_intraday_feature_reads':1,'model_fits':0,'native_accounts':0}
(card/'preflight_result.json').write_text(json.dumps(preflight,indent=2)+'\n')
state={'status':'fit_started','plan_sha256':preflight['plan_sha256'],'fit_attempts':0,'fit_completed':0,'bootstrap_attempts':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
labels=['cash_advantage_H5','SOXL_relative_H5'];metrics={};gates={};models={};scored=pd.DataFrame({'signal_date':pd.DatetimeIndex(he.index.get_level_values('datetime'))});full=pd.DataFrame({'signal_date':pd.DatetimeIndex(held_all.index.get_level_values('datetime'))});replicates={};bootstrap_indices=None
for label in labels:
 ytr=tr[('label',label)].to_numpy(float);yhe=he[('label',label)].to_numpy(float)
 assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
 base=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
 candidate=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
 for model,x in ((base,xtr),(candidate,xtr_new)):
  state['fit_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
  model.fit(x,ytr);state['fit_completed']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 p0=base.predict(xhe);p1=candidate.predict(xhe_new);diff=(yhe-p0)**2-(yhe-p1)**2
 scored[label+'_target']=yhe;scored[label+'_baseline']=p0;scored[label+'_candidate']=p1;scored[label+'_paired_improvement']=diff
 coef=float(candidate.named_steps['ridge'].coef_[-1]);models[label]={'baseline':base,'candidate':candidate}
 half=[]
 for start,end in plan['cohorts']['held_halves']:
  z=scored.loc[scored.signal_date.between(start,end),label+'_paired_improvement'];half.append({'start':start,'end':end,'rows':len(z),'mean_improvement':float(z.mean())})
 annual=[]
 for year in plan['cohorts']['complete_held_years']:
  z=scored.loc[scored.signal_date.dt.year.eq(year),label+'_paired_improvement'];annual.append({'year':year,'rows':len(z),'mean_improvement':float(z.mean())})
 data=full.merge(scored[['signal_date',label+'_paired_improvement']],on='signal_date',how='left',validate='one_to_one')
 assert len(data)==944 and int(data[label+'_paired_improvement'].notna().sum())==940
 state['bootstrap_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 b=plan['information_gate']['bootstrap'];interval,idx,reps=full_calendar_bootstrap(data[label+'_paired_improvement'].to_numpy(float),mean_block=b['mean_block_sessions'],resamples=b['draws_each'],seed=b['seed'],alpha=b['one_sided_alpha_each'])
 assert interval['valid_replicates']==b['draws_each']
 if bootstrap_indices is None:bootstrap_indices=idx
 else:assert np.array_equal(bootstrap_indices,idx)
 replicates[label]=reps
 metrics[label]={'baseline_held_mse':float(np.mean((yhe-p0)**2)),'candidate_held_mse':float(np.mean((yhe-p1)**2)),'paired_mean_improvement':float(diff.mean()),'added_standardized_coefficient':coef,'halves':half,'complete_years':annual,'bootstrap':interval}
 gates[label]={'coefficient_negative':coef<0,'full_held_positive':diff.mean()>0,'both_halves_positive':all(x['mean_improvement']>0 for x in half),'both_complete_years_positive':all(x['mean_improvement']>0 for x in annual),'corrected_lower_positive':interval['lower_bound']>0}
full=full.merge(scored,on='signal_date',how='left',validate='one_to_one');assert len(full)==944
joint_pass=all(all(g.values()) for g in gates.values())
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight)
 rec.save_objects(artifact_path='models',**{k+'_baseline':v['baseline'] for k,v in models.items()},**{k+'_candidate':v['candidate'] for k,v in models.items()})
 rec.save_objects(artifact_path='predictions',scored=scored)
 rec.save_objects(artifact_path='analysis',full_held_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=bootstrap_indices,joint_information_pass=joint_pass)
 rec.log_metrics(model_fits=state['fit_completed'],native_accounts=0,joint_information_pass=int(joint_pass),cash_improvement=metrics['cash_advantage_H5']['paired_mean_improvement'],relative_improvement=metrics['SOXL_relative_H5']['paired_mean_improvement'])
 recorder_id,experiment_id=rec.id,str(rec.experiment_id)
state['status']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':'conditional_dual_H5_information_pass_source_PIT_pending' if joint_pass else 'not_promoted_fixed_0945_cross_response_dual_H5_information_failed','objective_achieved':False,'plan_sha256':preflight['plan_sha256'],'preflight':preflight,'metrics':metrics,'gates':gates,'joint_information_pass':joint_pass,'new_model_fits':state['fit_completed'],'new_bootstraps':state['bootstrap_attempts'],'new_native_accounts':0,'held_history_reused':True,'historical_first_receipt_qualified':False,'independent_forward_qualified_days':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['Information MSE is not a fee-net own-state account.','Historical first receipt and published-affine source vintage remain unqualified.','Reused 2023-2026 market history is development only, not independent validation.']}
def json_ready(value):
 if isinstance(value,dict):return {str(k):json_ready(v) for k,v in value.items()}
 if isinstance(value,(list,tuple)):return [json_ready(v) for v in value]
 if isinstance(value,np.generic):return value.item()
 return value
summary=json_ready(summary)
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':remote};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'preflight':preflight,'metrics':metrics,'gates':gates,'recorder_id':recorder_id,'nas_sha256':remote},ensure_ascii=False,indent=2))


## 中文结论

训练 1054、评价成熟 940。现金优势 H5 平均 MSE 改善 +3.40425e-6，系数 -0.00102921，但区块下界 -5.69369e-6；SOXL 相对 H5 改善 +1.57657e-5，系数 +0.00193326 与预定负号相反，下界 -2.94656e-5。联合门失败，不晋级，不改变符号、倍数或窗口。

首轮传参错误耗费2次拟合；固定重跑4次拟合与2次区块检验写入 FINISHED Recorder，随后 JSON 布尔序列化失败。最终 summary/NAS 从保存 Recorder 只读恢复，累计6拟合、0账户。原件来源 PIT、0bp 费用后 SOXX 门、2/5bp 与独立前向均未通过。